In [1]:
import numpy as np 
from dgps import sbm_data, generate_gaussian_data
from methods import diffusion_maps, normalised_laplacian

from scipy.stats import multiscale_graphcorr

In [2]:
rng = np.random.default_rng(1)
A, X = sbm_data(10, rng=rng, correlated=True)

In [7]:
A, B, X, Y = generate_gaussian_data(n=5, k=2, sigma=0, edge_var=1, rng=rng, link_function='logistic')

In [9]:
L = normalised_laplacian(A)
U = diffusion_maps(L, k=2, t=1)

In [ ]:
def MGC_simulation(n, nsim, dgp='SBM', rng=None, k=2, t=1, alpha=0.05,
                   sigma=0, edge_var=1):
    if rng is None:
        rng = np.random.default_rng()

    res = []
    rejected = 0
    for i in range(nsim):
        if dgp == 'SBM':
            A, X = sbm_data(n, rng=rng, correlated=True)
            L = normalised_laplacian(A)
            U = diffusion_maps(L, k=k, t=t)
            embedded_A = U[0]
            embedded_B = X
        elif dgp == 'gaussian':
            A, B, Z, X = generate_gaussian_data(n, k, sigma=sigma, edge_var=edge_var, 
                                                rng=rng, link_function='logistic')
            LA = normalised_laplacian(A)
            LB = normalised_laplacian(B)
            UA = diffusion_maps(LA, k=k, t=t)
            UB = diffusion_maps(LB, k=k, t=t)
            embedded_A = UA[0]
            embedded_B = UB[0]

        out_mgc = multiscale_graphcorr(embedded_A, embedded_B)
        res.append(out_mgc)         

        if out_mgc.pvalue < alpha:
            rejected += 1

    return res, rejected

In [22]:
rng = np.random.default_rng(1)
res, rejected = MGC_simulation(n=10, dgp='SBM', nsim=10, rng=rng, k=1, t=1, alpha=0.05)
print(rejected)

1


In [23]:
res, rejected = MGC_simulation(n=20, dgp='gaussian', nsim=10, rng=rng, k=3, t=1, alpha=0.05, 
                               sigma=0.99, edge_var=10)
print(rejected)

7


In [ ]:
from methods import solve_independent, rv_coefficient, rv_coef_test

def rv_simulation(n, nsim, dgp='SBM', rng=None, k=2, 
                  t=1, alpha=0.05, sigma=0, edge_var=1):

    if rng is None:
        rng = np.random.default_rng()

    p_vals = []
    for i in range(nsim):
        if dgp == 'SBM':
            A, B = sbm_data(n, rng=rng, correlated=True)
        
        p_val = rv_coef_test(A, B, n_perm=n_perm, rng=rng, k=k)
        p_vals.append(p_val)

    return p_vals

rng = np.random.default_rng(1)
n = 50
k = 5
edge_var = 5
sigma = 0

A, B, Z, X = generate_gaussian_data(n, k, sigma, edge_var=edge_var, rng=rng)

Zhat, evals_A = solve_independent(A, k=k)
Xhat, evals_B = solve_independent(B, k=k)
rv_est = rv_coefficient(Zhat, Xhat)
rv_coefficient(A, B), rv_coefficient(Zhat, Xhat), rv_coefficient(Z, X)
n_perm = 100
rv_distr = []
for i in range(n_perm):
    perm = rng.permutation(n)
    X_perm = X[perm, :]
    rv_perm = rv_coefficient(X_perm, Z)
    rv_distr.append(rv_perm)
fig, ax = plt.subplots()
ax.hist(rv_distr)
ax.axvline(rv_est, color='red')
plt.show()